In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/jonathantran/Desktop/nba_economics")
STAGING_DIR = PROJECT_ROOT / "data" / "staging"

# Inspect consolidated award sources, avoiding yearly raw files.
award_files = sorted(
    path for path in (STAGING_DIR / "awards").glob("*.csv")
)

for path in award_files:
    df = pd.read_csv(path, low_memory=False)

    print(f"\nFILE: {path.name}")
    print("SHAPE:", df.shape)
    print("COLUMNS:", df.columns.tolist())
    print("DUPLICATE ROWS:", df.duplicated().sum())

    display(df.head(3))


FILE: S14A_all_nba.csv
SHAPE: (390, 4)
COLUMNS: ['season', 'all_nba_team', 'player_name', 'player_position']
DUPLICATE ROWS: 0


,season,all_nba_team,player_name,player_position
0,2025-2026,1,Nikola Jokić,C
1,2025-2026,2,Jaylen Brown,F
2,2025-2026,3,Jalen Duren,C



FILE: defense_history.csv
SHAPE: (263, 10)
COLUMNS: ['team_selection', 'player_name', 'age', 'team', 'points_won', 'points_max', 'vote_share', 'position', 'season', 'year']
DUPLICATE ROWS: 0


,team_selection,player_name,age,team,points_won,points_max,vote_share,position,season,year
0,1st,Dikembe Mutombo,34,TOT,43,58,0.741,C,2000-2001,2001
1,1st,Tim Duncan,24,SAS,40,58,0.690,F,2000-2001,2001
2,1st,Jason Kidd,27,PHO,39,58,0.672,G,2000-2001,2001



FILE: dpoy_history.csv
SHAPE: (426, 11)
COLUMNS: ['voting_rank', 'player_name', 'age', 'team', 'first_place_votes', 'points_won', 'points_max', 'vote_share', 'season', 'year', 'award_winner']
DUPLICATE ROWS: 0


...

,team_selection,player_name,age,team,points_won,points_max,vote_share,first_team_votes,second_team_votes,season,year
0,1st,Mike Miller,20,ORL,56,56,1.000,28,0,2000-2001,2001
1,1st,Kenyon Martin,23,NJN,54,56,0.964,27,0,2000-2001,2001
2,1st,Marc Jackson,26,GSW,51,56,0.911,24,3,2000-2001,2001



FILE: roy_history.csv
SHAPE: (198, 11)
COLUMNS: ['voting_rank', 'player_name', 'age', 'team', 'first_place_votes', 'points_won', 'points_max', 'vote_share', 'season', 'year', 'award_winner']
DUPLICATE ROWS: 0


,voting_rank,player_name,age,team,first_place_votes,points_won,points_max,vote_share,season,year,award_winner
0,1,Mike Miller,20,ORL,75,75,124,0.605,2000-2001,2001,True
1,2,Kenyon Martin,23,NJN,36,36,124,0.290,2000-2001,2001,False
2,3,Marc Jackson,26,GSW,7,7,124,0.056,2000-2001,2001,False



FILE: s14c_all_star_selections_2001_2026.csv
SHAPE: (681, 10)
COLUMNS: ['position', 'player_name', 'team', 'all_star_appearances', 'votes', 'selection_type', 'player_status', 'year', 'season', 'conference']
DUPLICATE ROWS: 0


,position,player_name,team,all_star_appearances,votes,selection_type,player_status,year,season,conference
0,G,Allen Iverson,Philadelphia 76ers,2,1508142.0,starter,NaN,2001,2000-2001,east
1,G,Tracy McGrady,Orlando Magic,1,827122.0,starter,NaN,2001,2000-2001,east
2,F,Vince Carter,Toronto Raptors,2,1717687.0,starter,NaN,2001,2000-2001,east


In [2]:
import re
import unicodedata

# Load canonical players and seasons.
players = pd.read_csv(
    PROJECT_ROOT / "data/canonical/players.csv"
)

seasons = pd.read_csv(
    PROJECT_ROOT / "data/canonical/seasons.csv"
)

# Load consolidated award files.
award_sources = {
    "all_nba": "S14A_all_nba.csv",
    "all_defensive": "defense_history.csv",
    "dpoy": "dpoy_history.csv",
    "mvp": "mvp_history.csv",
    "all_rookie": "rookie_history.csv",
    "roy": "roy_history.csv",
    "all_star": "s14c_all_star_selections_2001_2026.csv",
}

award_frames = {}

def normalize_text(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )
    value = value.casefold()
    value = re.sub(r"[^a-z0-9]+", " ", value)

    return " ".join(value.split())


# Prepare canonical player-name lookup.
player_lookup = players[
    ["player_id", "player_name"]
].copy()

player_lookup["player_key"] = (
    player_lookup["player_name"].map(normalize_text)
)

# Identify names shared by multiple canonical players.
ambiguous_names = (
    player_lookup.groupby("player_key")["player_id"]
    .nunique()
)

ambiguous_names = set(
    ambiguous_names[ambiguous_names > 1].index
)

# Keep only unambiguous automatic mappings.
player_lookup = player_lookup[
    ~player_lookup["player_key"].isin(ambiguous_names)
][["player_key", "player_id"]].drop_duplicates()


for award_type, filename in award_sources.items():

    df = pd.read_csv(
        STAGING_DIR / "awards" / filename
    )

    df["player_key"] = (
        df["player_name"].map(normalize_text)
    )

    # Match to canonical player IDs.
    df = df.merge(
        player_lookup,
        on="player_key",
        how="left",
        validate="many_to_one"
    )

    award_frames[award_type] = df

    print(f"\n{award_type.upper()}")
    print("-" * 45)

    print("Records:", len(df))
    print("Seasons:", df["season"].nunique())
    print("Matched player IDs:", df["player_id"].notna().sum())
    print("Unmatched player IDs:", df["player_id"].isna().sum())

    # Check repeated names within the same award and season.
    print(
        "Duplicate player-season records:",
        df.duplicated(["player_name", "season"]).sum()
    )

    # Inspect the categorical values relevant to this award.
    for column in [
        "team_selection",
        "all_nba_team",
        "selection_type",
        "award_winner"
    ]:
        if column in df.columns:
            print(f"{column}:", df[column].value_counts(
                dropna=False
            ).to_dict())

    # Display unmatched player names.
    unmatched = (
        df.loc[df["player_id"].isna(), "player_name"]
        .value_counts()
        .head(15)
    )

    if not unmatched.empty:
        print("Top unmatched names:")
        display(unmatched)


ALL_NBA
---------------------------------------------
Records: 390
Seasons: 26
Matched player IDs: 385
Unmatched player IDs: 5
Duplicate player-season records: 0
all_nba_team: {1: 130, 2: 130, 3: 130}
Top unmatched names:


player_name
Jimmy Butler    5
Name: count, dtype: int64


ALL_DEFENSIVE
---------------------------------------------
Records: 263
Seasons: 26
Matched player IDs: 257
Unmatched player IDs: 6
Duplicate player-season records: 0
team_selection: {'1st': 132, '2nd': 131}
Top unmatched names:


player_name
Jimmy Butler       5
Robert Williams    1
Name: count, dtype: int64


DPOY
---------------------------------------------
Records: 426
Seasons: 26
Matched player IDs: 419
Unmatched player IDs: 7
Duplicate player-season records: 0
award_winner: {False: 400, True: 26}
Top unmatched names:


...

player_name
Marcus Williams    1
GG Jackson II      1
Name: count, dtype: int64


ROY
---------------------------------------------
Records: 198
Seasons: 26
Matched player IDs: 197
Unmatched player IDs: 1
Duplicate player-season records: 0
award_winner: {False: 172, True: 26}
Top unmatched names:


player_name
J.R. Smith    1
Name: count, dtype: int64


ALL_STAR
---------------------------------------------
Records: 681
Seasons: 26
Matched player IDs: 655
Unmatched player IDs: 26
Duplicate player-season records: 0
selection_type: {'reserve': 421, 'starter': 260}
Top unmatched names:


player_name
Jimmy Butler           6
Amare Stoudemire       3
Jayson TatumST         2
Ron Artest             1
Allen IversonDNP       1
Dwyane Wade (C)        1
Chris Paul (C)         1
LaMarcus AldridgeST    1
Klay ThompsonST        1
James HardenST         1
Joel EmbiidCOVID       1
Ben SimmonsCOVID       1
Zion WilliamsonST      1
Joel EmbiidST          1
Ja MorantST            1
Name: count, dtype: int64

In [3]:
# Gather all currently unmatched award observations.
unmatched_awards = pd.concat(
    [
        df.loc[
            df["player_id"].isna(),
            ["player_name", "season"]
        ].assign(award_type=award_type)
        for award_type, df in award_frames.items()
    ],
    ignore_index=True
)

print("ALL UNMATCHED AWARD OBSERVATIONS")
display(
    unmatched_awards.sort_values(
        ["player_name", "season"]
    )
)

# Search C03 for likely alternate player names.
search_terms = [
    "butler", "stoudemire", "williams",
    "jackson", "smith", "artest",
    "peace", "hilario", "tatum",
    "iverson", "embiid", "wade",
    "aldridge", "thompson", "harden",
    "simmons", "morant"
]

possible_matches = players[
    players["player_name"].str.contains(
        "|".join(search_terms),
        case=False,
        na=False,
        regex=True
    )
][["player_id", "player_name"]]

print("\nPOSSIBLE CANONICAL PLAYER MATCHES")
display(
    possible_matches.sort_values("player_name")
)

ALL UNMATCHED AWARD OBSERVATIONS


,player_name,season,award_type
28,Allen IversonDNP,2009-2010,all_star
25,Amare Stoudemire,2004-2005,all_star
26,Amare Stoudemire,2006-2007,all_star
27,Amare Stoudemire,2007-2008,all_star
48,Bam AdebayoST,2023-2024,all_star
40,Ben SimmonsCOVID,2020-2021,all_star
30,Chris Paul (C),2012-2013,all_star
29,Dwyane Wade (C),2012-2013,all_star
22,GG Jackson II,2023-2024,all_rookie
23,J.R. Smith,2004-2005,roy



POSSIBLE CANONICAL PLAYER MATCHES


,player_id,player_name
1783,1628935,Aaron Jackson
2033,1630174,Aaron Nesmith
267,1425,Aaron Williams
1559,1626210,Alan Williams
216,947,Allen Iverson
...,...,...
1585,1627735,Wade Baldwin IV
248,1005,Walt Williams
1832,1629015,Zhaire Smith
2129,1630533,Ziaire Williams


In [5]:
# Load C05 to help resolve ambiguous player identities.
player_team_seasons = pd.read_csv(
    PROJECT_ROOT / "data/canonical/player_team_seasons.csv"
)



# STEP 1: Find the canonical names for our unresolved players.

target_terms = [
    "butler", "williams", "jackson",
    "smith", "stoudemire", "artest",
    "peace"
]

canonical_candidates = players[
    players["player_name"].str.contains(
        "|".join(target_terms),
        case=False,
        na=False
    )
][["player_id", "player_name"]]

print("CANONICAL NAME CANDIDATES")

display(
    canonical_candidates.sort_values("player_name")
)


# STEP 2: Inspect Marcus Williams in the 2006-2007 season.

marcus_ids = players.loc[
    players["player_name"] == "Marcus Williams",
    "player_id"
]

print("\nMARCUS WILLIAMS — 2006-2007")

display(
    player_team_seasons.loc[
        (player_team_seasons["player_id"].isin(marcus_ids))
        & (player_team_seasons["season"] == "2006-2007"),
        ["player_id", "franchise_id", "season"]
    ]
)


# STEP 3: Test cleaning of All-Star annotations.

all_star = award_frames["all_star"].copy()

# Remove only recognized annotations at the end of names.
all_star["clean_player_name"] = (
    all_star["player_name"]
    .str.replace(
        r"(ST|DNP|COVID|NOTE|\s*\(C\))$",
        "",
        regex=True
    )
    .str.strip()
)

all_star["clean_player_key"] = (
    all_star["clean_player_name"].map(normalize_text)
)

# Reuse the unambiguous canonical player lookup.
all_star_test = all_star[
    ["player_name", "clean_player_name", "clean_player_key"]
].merge(
    player_lookup.rename(
        columns={"player_key": "clean_player_key"}
    ),
    on="clean_player_key",
    how="left",
    validate="many_to_one"
)

print("\nALL-STAR MATCHING AFTER ANNOTATION CLEANUP")

print(
    "Matched:",
    all_star_test["player_id"].notna().sum(),
    "of",
    len(all_star_test)
)

print("\nREMAINING UNMATCHED ALL-STAR NAMES")

display(
    all_star_test.loc[
        all_star_test["player_id"].isna(),
        ["player_name", "clean_player_name"]
    ].drop_duplicates()
)

CANONICAL NAME CANDIDATES


,player_id,player_name
1783,1628935,Aaron Jackson
2033,1630174,Aaron Nesmith
267,1425,Aaron Williams
1559,1626210,Alan Williams
2297,1631214,Alondes Williams
...,...,...
2311,1631246,Vince Williams Jr.
248,1005,Walt Williams
1832,1629015,Zhaire Smith
2129,1630533,Ziaire Williams



MARCUS WILLIAMS — 2006-2007


,player_id,franchise_id,season
2957,200766,1610612751,2006-2007



ALL-STAR MATCHING AFTER ANNOTATION CLEANUP
Matched: 671 of 681

REMAINING UNMATCHED ALL-STAR NAMES


,player_name,clean_player_name
88,Ron Artest,Ron Artest
114,Amare Stoudemire,Amare Stoudemire
369,Jimmy Butler,Jimmy Butler


In [6]:
# Show only the specific canonical names we need.
terms = [
    "Jimmy", "Butler", "Stoud", "Amar",
    "Artest", "Peace", "Robert Williams",
    "Jackson", "Smith"
]

candidates = players[
    players["player_name"].str.contains(
        "|".join(terms),
        case=False,
        na=False
    )
].copy()

# Narrow the output to the relevant player-name patterns.
candidates = candidates[
    candidates["player_name"].str.contains(
        r"Jimmy|Stoud|Amar|Artest|Peace|Robert Williams|GG Jackson|J\.?R\.? Smith",
        case=False,
        regex=True,
        na=False
    )
]

display(
    candidates[["player_id", "player_name"]]
    .sort_values("player_name")
    .to_string(index=False)
)

" player_id         player_name\n      2405   Amar'e Stoudemire\n   1641735        Amari Bailey\n   1642873      Amari Williams\n       757    Damon Stoudamire\n   1641713          GG Jackson\n      2747            JR Smith\n   1631123      Jamaree Bouyea\n    200081        Jamario Moon\n    202710    Jimmy Butler III\n    200746   LaMarcus Aldridge\n      1885          Lamar Odom\n    203934     Lamar Patterson\n   1630205       Lamar Stevens\n      1897   Metta World Peace\n   1629057 Robert Williams III\n    101136    Salim Stoudamire\n    202396     Samardo Samuels\n      2447          Tamar Slay\n   1641739      Toumani Camara"

In [7]:
# Explicit aliases based on confirmed C03 identities.
name_aliases = {
    "Jimmy Butler": "Jimmy Butler III",
    "Robert Williams": "Robert Williams III",
    "GG Jackson II": "GG Jackson",
    "J.R. Smith": "JR Smith",
    "Amare Stoudemire": "Amar'e Stoudemire",
    "Ron Artest": "Metta World Peace",
}

# Known All-Star annotations appended to player names.
all_star_suffixes = r"(ST|DNP|COVID|NOTE|\s*\(C\))$"

award_parts = []

for award_type, original_df in award_frames.items():

    df = original_df.drop(
        columns=["player_id", "player_key"],
        errors="ignore"
    ).copy()

    # Preserve original source identity.
    df = df.rename(
        columns={"player_name": "source_player_name"}
    )

    # Remove All-Star-specific name annotations.
    df["clean_player_name"] = df["source_player_name"]

    if award_type == "all_star":
        df["clean_player_name"] = (
            df["clean_player_name"]
            .str.replace(all_star_suffixes, "", regex=True)
            .str.strip()
        )

    # Apply only explicitly verified name aliases.
    df["clean_player_name"] = (
        df["clean_player_name"]
        .replace(name_aliases)
    )

    df["player_key"] = (
        df["clean_player_name"].map(normalize_text)
    )

    # Attach canonical player identifiers.
    df = df.merge(
        player_lookup,
        on="player_key",
        how="left",
        validate="many_to_one"
    )

    # Resolve the one confirmed ambiguous Marcus Williams record.
    if award_type == "all_rookie":
        marcus_mask = (
            df["source_player_name"].eq("Marcus Williams")
            & df["season"].eq("2006-2007")
        )

        df.loc[marcus_mask, "player_id"] = 200766

    df["award_type"] = award_type
    df["source_table"] = award_sources[award_type]

    # Standardize All-NBA selections to match other team awards.
    if award_type == "all_nba":
        df["team_selection"] = (
            df["all_nba_team"]
            .map({1: "1st", 2: "2nd", 3: "3rd"})
        )

    award_parts.append(df)


# Concatenate sources, retaining award-specific columns.
c08 = pd.concat(
    award_parts,
    ignore_index=True,
    sort=False
)

# Keep source player names, but remove temporary matching fields.
c08 = c08.drop(
    columns=[
        "player_key",
        "clean_player_name",
        "all_nba_team"
    ],
    errors="ignore"
)

print("C08 rows:", len(c08))
print("Matched:", c08["player_id"].notna().sum())
print("Unmatched:", c08["player_id"].isna().sum())

display(
    c08.groupby("award_type")
    .agg(
        records=("player_id", "size"),
        matched=("player_id", "count")
    )
)

C08 rows: 2570
Matched: 2570
Unmatched: 0


,records,matched
award_type,,
all_defensive,263,263
all_nba,390,390
all_rookie,260,260
all_star,681,681
dpoy,426,426
mvp,352,352
roy,198,198


In [8]:
CANONICAL_DIR = PROJECT_ROOT / "data" / "canonical"

# Validate canonical identity and season coverage.
assert c08["player_id"].notna().all(), "Unmatched players remain."
assert c08["season"].isin(seasons["season"]).all(), "Unknown seasons."
assert c08["player_id"].isin(players["player_id"]).all(), "Unknown player IDs."

# Each player should appear at most once per award category per season.
key_cols = ["player_id", "season", "award_type"]

duplicates = c08.loc[
    c08.duplicated(key_cols, keep=False),
    key_cols + ["source_player_name"]
]

display(duplicates)

assert duplicates.empty, "Duplicate player-season-award keys found."

# Check award winners separately from award voting candidates.
for award_type in ["mvp", "dpoy", "roy"]:
    subset = c08[c08["award_type"] == award_type]

    winner_counts = (
        subset.groupby("season")["award_winner"]
        .sum()
    )

    assert winner_counts.eq(1).all(), (
        f"Unexpected winner counts in {award_type}"
    )

# Finalize player ID type and sort.
c08["player_id"] = c08["player_id"].astype("int64")

c08 = c08.sort_values(
    ["season", "award_type", "player_id"]
).reset_index(drop=True)

# Export canonical table.
c08.to_csv(
    CANONICAL_DIR / "player_awards.csv",
    index=False
)

print("\nC08 VALIDATION PASSED")
print("---------------------------------------------")
print("Canonical award rows:", len(c08))
print("Seasons represented:", c08["season"].nunique())
print("Unique players:", c08["player_id"].nunique())
print("Award categories:", c08["award_type"].nunique())
print("Files exported successfully.")

,player_id,season,award_type,source_player_name



C08 VALIDATION PASSED
---------------------------------------------
Canonical award rows: 2570
Seasons represented: 26
Unique players: 438
Award categories: 7
Files exported successfully.
